# Blind2Unblind — Implementation / 구현

**Paper**: Wang, Z., Liu, J., Li, G., & Han, H. (2022). "Blind2Unblind: Self-Supervised Image Denoising with Visible Blind Spots", *CVPR 2022*. [DOI: 10.1109/CVPR52688.2022.00207]

## Overview / 개요

이 노트북은 Blind2Unblind의 두 가지 핵심 구성요소를 직접 구현한다:
1. **Global-aware mask mapper** \$h(\cdot)\$ — 2×2 셀 4-mask 볼륨을 생성하고 *blind-spot 위치만* 단일 평면으로 sampling.
2. **Re-visible loss** \$\mathcal L_{\rm rev} = \|h(f_\theta(\Omega_y)) + \lambda \hat f_\theta(y) - (\lambda+1) y\|^2\$.

`cameraman` 영상에 가우시안 잡음을 더한 뒤 (a) Noise2Void(blind-only) 학습 vs (b) Blind2Unblind(re-visible) 학습을 비교.

This notebook implements:
1. The **global-aware mask mapper** \$h\$ that creates the 4-mask volume \$\Omega_y\$ and gathers blind-spot pixels.
2. The **re-visible loss** with stop-gradient on the non-blind branch.

We compare a Noise2Void-style blind-only loss against the Blind2Unblind re-visible loss on a noisy `cameraman` image.


In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from numpy.typing import NDArray

plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['font.size'] = 10

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
rng = np.random.default_rng(42)
print('Device:', DEVICE)


---

## Part 1: Global-aware mask mapper / 글로벌 마스크 매퍼

\$2\times 2\$ 셀당 한 픽셀씩 가린 4가지 마스크 \$\Omega_y^{ij}\$를 만들어 채널로 stacking. 결과 볼륨 \$\Omega_y \in \mathbb R^{B\times 4 \times H\times W}\$.


In [ ]:
def make_masked_volume(y: torch.Tensor) -> torch.Tensor:
    """Build the 4-channel masked volume Omega_y for a 2x2 cell.

    For each cell position (i, j) in {0, 1}^2, mask out pixels whose
    (p % 2, q % 2) == (i, j) and stack along channel.

    Args:
        y: [B, 1, H, W] noisy input (H, W must be even).

    Returns:
        [B, 4, H, W] masked volume.
    """
    B, _, H, W = y.shape
    out = torch.zeros(B, 4, H, W, device=y.device, dtype=y.dtype)
    pp, qq = torch.meshgrid(torch.arange(H, device=y.device),
                            torch.arange(W, device=y.device), indexing='ij')
    for k, (i, j) in enumerate([(0, 0), (0, 1), (1, 0), (1, 1)]):
        mask = ~((pp % 2 == i) & (qq % 2 == j))  # True where we keep the pixel
        out[:, k] = y[:, 0] * mask.to(y.dtype)
    return out


def mask_mapper(volume: torch.Tensor) -> torch.Tensor:
    """h(.) — gather denoised values at blind-spot positions onto a single plane.

    For each pixel (p, q), pick channel k corresponding to (p % 2, q % 2) — i.e.,
    the channel where (p, q) was the blind spot.

    Args:
        volume: [B, 4, H, W] denoised volume f_theta(Omega_y).

    Returns:
        [B, 1, H, W] global denoised plane h(f_theta(Omega_y)).
    """
    B, _, H, W = volume.shape
    pp, qq = torch.meshgrid(torch.arange(H, device=volume.device),
                            torch.arange(W, device=volume.device), indexing='ij')
    out = torch.zeros(B, 1, H, W, device=volume.device, dtype=volume.dtype)
    for k, (i, j) in enumerate([(0, 0), (0, 1), (1, 0), (1, 1)]):
        sel = (pp % 2 == i) & (qq % 2 == j)  # blind-spot positions for channel k
        out[:, 0] = torch.where(sel, volume[:, k], out[:, 0])
    return out


# Smoke test
y = torch.arange(16, dtype=torch.float32).reshape(1, 1, 4, 4)
print('y =\n', y[0, 0])
vol = make_masked_volume(y)
print('Channel 0 (mask (0,0)) =\n', vol[0, 0])
print('Channel 1 (mask (0,1)) =\n', vol[0, 1])
# Mask mapper should reproduce y if every channel == y
identity_test = mask_mapper(y.expand(-1, 4, -1, -1).clone())
print('Identity check (mapper of repeated y):', torch.allclose(identity_test, y))


---

## Part 2: Build noisy cameraman / 잡음 cameraman 생성


In [ ]:
try:
    from skimage import data
    cameraman = data.camera().astype(np.float32) / 255.0
except Exception:
    sz = 256
    yy, xx = np.meshgrid(np.linspace(-1, 1, sz), np.linspace(-1, 1, sz), indexing='ij')
    cameraman = (0.5 + 0.4 * np.exp(-(xx**2 + yy**2) / 0.4)
                 + 0.2 * (np.abs(xx) < 0.3)).astype(np.float32)

# Downsample to 128x128 for tractable CPU training (paper-quality demo, not full benchmark)
SIZE = 128
cameraman = cameraman[::cameraman.shape[0] // SIZE, ::cameraman.shape[1] // SIZE][:SIZE, :SIZE]

SIGMA = 25 / 255.0
y_noisy_np = cameraman + SIGMA * rng.standard_normal(cameraman.shape).astype(np.float32)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(cameraman, cmap='gray', vmin=0, vmax=1); axes[0].set_title('Clean'); axes[0].axis('off')
axes[1].imshow(y_noisy_np, cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'Noisy (sigma=25/255)'); axes[1].axis('off')
plt.tight_layout(); plt.show()

def psnr(a: np.ndarray, b: np.ndarray) -> float:
    return float(10.0 * np.log10(1.0 / max(np.mean((a - b) ** 2), 1e-12)))
print(f'Image size: {cameraman.shape}')
print(f'Noisy PSNR: {psnr(y_noisy_np, cameraman):.2f} dB')

---

## Part 3: Tiny U-Net backbone / 작은 U-Net 백본

Receptive field가 작은 4-level mini U-Net. Production B2U는 modified U-Net을 사용.


In [ ]:
class MiniUNet(nn.Module):
    """Minimal U-Net (3 levels). Takes 1- or 4-channel input, outputs same channel."""

    def __init__(self, in_ch: int = 1, out_ch: int = 1, base: int = 16) -> None:
        super().__init__()
        self.e1 = nn.Sequential(nn.Conv2d(in_ch, base, 3, padding=1), nn.ReLU(inplace=True),
                                nn.Conv2d(base, base, 3, padding=1), nn.ReLU(inplace=True))
        self.e2 = nn.Sequential(nn.Conv2d(base, 2*base, 3, padding=1), nn.ReLU(inplace=True),
                                nn.Conv2d(2*base, 2*base, 3, padding=1), nn.ReLU(inplace=True))
        self.e3 = nn.Sequential(nn.Conv2d(2*base, 4*base, 3, padding=1), nn.ReLU(inplace=True),
                                nn.Conv2d(4*base, 4*base, 3, padding=1), nn.ReLU(inplace=True))
        self.up2 = nn.ConvTranspose2d(4*base, 2*base, 2, stride=2)
        self.d2 = nn.Sequential(nn.Conv2d(4*base, 2*base, 3, padding=1), nn.ReLU(inplace=True),
                                nn.Conv2d(2*base, 2*base, 3, padding=1), nn.ReLU(inplace=True))
        self.up1 = nn.ConvTranspose2d(2*base, base, 2, stride=2)
        self.d1 = nn.Sequential(nn.Conv2d(2*base, base, 3, padding=1), nn.ReLU(inplace=True),
                                nn.Conv2d(base, out_ch, 3, padding=1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        e1 = self.e1(x)
        e2 = self.e2(F.max_pool2d(e1, 2))
        e3 = self.e3(F.max_pool2d(e2, 2))
        u2 = self.up2(e3)
        d2 = self.d2(torch.cat([u2, e2], dim=1))
        u1 = self.up1(d2)
        return self.d1(torch.cat([u1, e1], dim=1))


# Lift the 4-channel volume forward through a single-scale model:
# Production B2U processes the volume as a stacked batch — we do the same here.


---

## Part 4: Train blind-only (N2V analog) vs re-visible (B2U) / 학습

두 손실:

1. **Blind only** (N2V analog): 단일 \$\Omega_y\$ mask 사용, 손실은 mapper 출력 vs \$y\$:
   \[\mathcal L = \|h(f_\theta(\Omega_y)) - y\|^2.\]

2. **Re-visible** (B2U): non-blind branch 추가:
   \[\mathcal L = \|h(f_\theta(\Omega_y)) + \lambda \hat f_\theta(y) - (\lambda+1) y\|^2 + \eta \|h(f_\theta(\Omega_y)) - y\|^2.\]
   Stop-gradient on \$\hat f_\theta(y)\$.


In [ ]:
y_t = torch.from_numpy(y_noisy_np).unsqueeze(0).unsqueeze(0).to(DEVICE)
x_t = torch.from_numpy(cameraman).unsqueeze(0).unsqueeze(0).to(DEVICE)


def train_b2u(loss_kind: str, n_iter: int = 300, lr: float = 1e-3,
              eta: float = 1.0, lam_init: float = 2.0, lam_final: float = 20.0,
              ) -> tuple[MiniUNet, list[float]]:
    """Train MiniUNet with either 'blind' (N2V-like) or 'rev' (B2U) loss.

    The B2U variant linearly anneals lambda from lam_init to lam_final.
    """
    net = MiniUNet(in_ch=1, out_ch=1, base=16).to(DEVICE)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    curve: list[float] = []
    for it in range(n_iter):
        opt.zero_grad()
        omega_y = make_masked_volume(y_t)
        vol_in = omega_y.permute(1, 0, 2, 3).contiguous()
        vol_out = net(vol_in)
        denoised_vol = vol_out.permute(1, 0, 2, 3).contiguous()
        h_out = mask_mapper(denoised_vol)

        if loss_kind == 'blind':
            loss = F.mse_loss(h_out, y_t)
        elif loss_kind == 'rev':
            with torch.no_grad():
                f_y = net(y_t)
            t = it / max(n_iter - 1, 1)
            lam = lam_init + (lam_final - lam_init) * t
            target = (lam + 1.0) * y_t
            l_rev = F.mse_loss(h_out + lam * f_y, target)
            l_reg = F.mse_loss(h_out, y_t)
            loss = l_rev + eta * l_reg
        else:
            raise ValueError(loss_kind)
        loss.backward(); opt.step()
        if it % 20 == 0:
            with torch.no_grad():
                pred = net(y_t).clamp(0, 1)
                mse = F.mse_loss(pred, x_t).item()
                curve.append(mse)
    return net, curve


print('Training blind-only ...'); net_blind, c_blind = train_b2u('blind')
print('Training re-visible ...'); net_rev,   c_rev   = train_b2u('rev')

iters = np.arange(len(c_blind)) * 20
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(iters, c_blind, 'C0-', label='Blind-only (N2V analog)')
ax.semilogy(iters, c_rev,   'C2-', label='Re-visible (B2U)')
ax.set_xlabel('Iteration'); ax.set_ylabel('MSE vs clean')
ax.legend(); ax.grid(True); plt.tight_layout(); plt.show()

---

## Part 5: Visual comparison / 시각 비교


In [ ]:
net_blind.eval(); net_rev.eval()
with torch.no_grad():
    pred_blind = net_blind(y_t).clamp(0, 1).cpu().numpy()[0, 0]
    pred_rev   = net_rev(y_t).clamp(0, 1).cpu().numpy()[0, 0]

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
panels = [(cameraman, 'Clean'),
          (y_noisy_np, f'Noisy ({psnr(y_noisy_np, cameraman):.1f} dB)'),
          (pred_blind, f'Blind-only ({psnr(pred_blind, cameraman):.1f} dB)'),
          (pred_rev,   f'B2U Re-visible ({psnr(pred_rev, cameraman):.1f} dB)')]
for ax, (img, title) in zip(axes, panels):
    ax.imshow(img, cmap='gray', vmin=0, vmax=1); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

print(f'Blind-only PSNR : {psnr(pred_blind, cameraman):.2f} dB')
print(f'B2U re-visible  : {psnr(pred_rev, cameraman):.2f} dB')
print(f'Improvement     : +{psnr(pred_rev, cameraman) - psnr(pred_blind, cameraman):.2f} dB')


---

## Part 6: Inspect the masked volume / 마스크된 볼륨 시각화

\$\Omega_y\$의 4채널을 시각화. 각 채널의 *checkerboard 빈 자리*가 blind spot.


In [ ]:
omega = make_masked_volume(y_t).cpu().numpy()[0]
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for k, ax in enumerate(axes):
    ax.imshow(omega[k, :32, :32], cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'$\\Omega_y$ ch {k}'); ax.axis('off')
plt.tight_layout(); plt.show()


---

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | This Notebook / 본 노트북 |
|---|---|---|
| Masked volume \$\Omega_y\$ | §4.1 with \$s=2\$ | `make_masked_volume()` |
| Mask mapper \$h\$ | §4.1 global-aware | `mask_mapper()` |
| Blind-only loss (N2V analog) | Eq. (1) | `train_b2u('blind')` |
| Re-visible loss \$\mathcal L_{\rm rev}\$ | Eq. (8) | `train_b2u('rev')` |
| Regulariser term | Eq. (12) | `loss = l_rev + eta * l_reg` |
| \$\lambda\$ annealing | \$\lambda_s=2\to\lambda_f=20\$ | linear schedule in `train_b2u` |
| Inference (single forward) | §3.1 | `net(y_t)` |

### Connections to other papers / 다른 논문과의 연결
- **Paper #21 R2R**: alternative single-image self-supervised approach using known noise statistics; complementary to B2U which assumes spatial smoothness instead.
- **Noise2Void (Krull+ 2018)**: predecessor; the blind-only branch here is essentially N2V with a 2×2 mask cell.
- **Modern U-Net denoisers**: B2U swaps in any U-Net variant (NAFNet, Restormer) without changing the loss machinery.

### Take-away / 결론
The mask mapper turns a checkerboard-masked volume into a coherent global denoised plane, and the re-visible loss exploits this to *re-introduce* the masked pixel via a non-blind branch under stop-gradient. On the cameraman test, B2U beats blind-only by a clear margin while keeping the same backbone — demonstrating the paper's central claim that the bottleneck of N2V was the loss, not the architecture.

마스크 매퍼는 체커보드 마스킹된 볼륨을 *일관된 전역 디노이즈 평면*으로 만들고, re-visible 손실은 stop-gradient된 non-blind 분기를 통해 가려졌던 픽셀을 *재투입*한다. cameraman 실험에서 B2U는 동일 백본의 blind-only 대비 명확한 차이를 보여 — 본 논문의 핵심 주장(N2V의 한계는 *아키텍처*가 아니라 *손실*에 있다)을 직접 확인.
